# Eliminación gaussiana para sistemas de ecuaciones lineales

Implementación del **método de eliminación gaussiana** para resolver sistemas de ecuaciones lineales utilizando Python y NumPy.

Este notebook reorganiza los ejercicios desarrollados durante el estudio de **Análisis Numérico** y se concentra únicamente en la eliminación gaussiana, eliminando apuntes de otros temas que no forman parte directa de este método.

### Objetivos

- Comprender el procedimiento de eliminación gaussiana.
- Transformar un sistema lineal en un sistema triangular superior.
- Obtener la solución mediante sustitución hacia atrás.
- Aplicar el método a un sistema de ecuaciones lineales.
- Analizar el manejo de un pivote inicial igual a cero mediante intercambio de filas.

**Tecnologías:** Python · NumPy

## 1. Fundamento del método

Un sistema de ecuaciones lineales puede escribirse en forma matricial como:

$Ax=b$

donde $A$ es la matriz de coeficientes, $x$ es el vector de incógnitas y $b$ es el vector de términos independientes.

La **eliminación gaussiana** es un método directo que transforma el sistema original en uno equivalente cuya matriz de coeficientes es triangular superior.

Para eliminar los elementos ubicados debajo de cada pivote se utiliza un factor de eliminación:

$$
\lambda=\frac{a_{ji}}{a_{ii}}
$$

y se actualizan las filas mediante operaciones elementales.

Una vez obtenida la matriz triangular superior, las incógnitas se calculan mediante **sustitución hacia atrás**:

$$
x_k=
\frac{
b_k-\sum_{j=k+1}^{n}a_{kj}x_j
}{
a_{kk}
}
$$

A diferencia de métodos iterativos como Jacobi o Gauss-Seidel, este procedimiento no requiere una tolerancia ni un vector inicial.

## 2. Implementación de la eliminación gaussiana

La siguiente función corresponde al **código original** utilizado para realizar la eliminación hacia adelante y posteriormente la sustitución hacia atrás.

Durante la eliminación se imprimen las matrices intermedias para visualizar cómo los elementos ubicados debajo de la diagonal principal se van anulando.

In [1]:
import numpy as np

def EliminacionGaussiana(Matriz, Vector):
  LongitudV=len(Vector)
  for i in range(LongitudV-1):
    for j in range(i+1, LongitudV):
      factor_lambda = Matriz[j, i]/Matriz[i, i]
      # Actualización
      Matriz[j] = Matriz[j] - factor_lambda * Matriz[i]
      Vector[j]=Vector[j] - factor_lambda * Vector[i]
      print(Matriz)
      print()
  x_sol = np.zeros_like(Vector)
    
  for k in range(LongitudV-1, -1, -1):
    x_sol[k] = (Vector[k] - np.dot(Matriz[k ,k+1:LongitudV], x_sol[k+1:LongitudV]))/Matriz[k, k]
  return x_sol

## 3. Aplicación a un sistema lineal

Se desea resolver el sistema:

$$
\begin{aligned}
-3x_1+2x_2+x_3 &= 2 \\
6x_1-8x_2-2x_3 &= 1 \\
x_1-x_2-2x_3 &= 3
\end{aligned}
$$

Su representación matricial es:

$$
A=
\begin{pmatrix}
-3 & 2 & 1\\
6 & -8 & -2\\
1 & -1 & -2
\end{pmatrix},
\qquad
b=
\begin{pmatrix}
2\\
1\\
3
\end{pmatrix}
$$

Se aplica la función de eliminación gaussiana definida anteriormente.

In [2]:
Matriz=np.array([[-3, 2, 1], [6, -8, -2], [1, -1, -2]], float)
Vector=np.array([2, 1, 3], float)

x_solucion=EliminacionGaussiana(Matriz, Vector)
print(f'La solución es {x_solucion}')

[[-3.  2.  1.]
 [ 0. -4.  0.]
 [ 1. -1. -2.]]

[[-3.          2.          1.        ]
 [ 0.         -4.          0.        ]
 [ 0.         -0.33333333 -1.66666667]]

[[-3.          2.          1.        ]
 [ 0.         -4.          0.        ]
 [ 0.          0.         -1.66666667]]

La solución es [-2.15 -1.25 -1.95]


### Resultado

La solución obtenida es aproximadamente:

$$
x=
\begin{pmatrix}
-2.15\\
-1.25\\
-1.95
\end{pmatrix}
$$

Al sustituir estos valores en el sistema original se recuperan los términos independientes, salvo pequeñas diferencias propias de la representación numérica en punto flotante.

## 4. Manejo de un pivote inicial igual a cero

La eliminación gaussiana requiere dividir por el elemento pivote $a_{ii}$. Si dicho elemento es cero, esa división no puede realizarse directamente.

En el siguiente ejercicio, la matriz comienza con un pivote igual a cero:

$$
A=
\begin{pmatrix}
0 & 2 & 1\\
6 & -8 & -2\\
1 & -1 & -2
\end{pmatrix}
$$

El código original detecta esta situación e **intercambia la fila actual con la fila siguiente** antes de continuar con la eliminación.

> **Importante:** esta implementación resuelve correctamente el ejercicio planteado, pero no corresponde a un pivoteo parcial general. Un algoritmo de pivoteo parcial completo buscaría, entre las filas disponibles, el elemento de mayor valor absoluto en la columna del pivote.

Se conserva la implementación original sin modificar su lógica.

In [3]:
import numpy as np

A = np.array([[0, 2, 1], [6, -8, -2],[1, -1, -2]], float)
b = np.array([2, 1, 3], float)

def gauss_pivoteo(A, b):
    n = len(b)
    
    # Eliminación hacia adelante
    for i in range(n-1):
        if A[i, i] == 0:
            aux = np.copy(A[i, :n])
            A[i, :n] = A[i+1, :n]
            A[i+1, :n] = aux

            aux1 = b[i]
            b[i] = b[i+1]
            b[i+1] = aux1

        for j in range(i+1, n):
            factor_lambda = A[j, i] / A[i, i]

            # actualización
            A[j] = A[j] - factor_lambda * A[i]
            b[j] = b[j] - factor_lambda * b[i]

    # Sustitución hacia atrás
    x_sol = np.zeros_like(b)
    for k in range(n-1, -1, -1):
        x_sol[k] = (b[k] - np.dot(A[k, k+1:n], x_sol[k+1:n])) / A[k, k]

    return x_sol

x_solucion = gauss_pivoteo(A, b)
print(f'La solución es {x_solucion}')


La solución es [ 1.95454545  1.68181818 -1.36363636]


### Resultado

Para este segundo sistema se obtiene aproximadamente:

$$
x=
\begin{pmatrix}
1.95454545\\
1.68181818\\
-1.36363636
\end{pmatrix}
$$

El intercambio de filas permite evitar la división inicial por cero y continuar con el proceso de eliminación.

## 5. Consideraciones sobre la implementación

Los códigos utilizados permiten observar directamente las etapas principales del método:

- eliminación hacia adelante;
- transformación de la matriz;
- sustitución hacia atrás;
- intercambio de filas cuando el pivote encontrado es cero.

La función `EliminacionGaussiana` modifica la matriz y el vector recibidos durante el proceso de eliminación, lo cual es coherente con la forma en que se desarrolló el ejercicio.

Para una implementación de propósito general podrían incorporarse estrategias adicionales de pivoteo, pero en este notebook se mantiene la lógica original de los códigos desarrollados.

## 6. Conclusiones

A partir de los ejercicios desarrollados se puede concluir que:

- La eliminación gaussiana permite resolver sistemas lineales transformándolos en sistemas triangulares superiores.
- Las operaciones elementales entre filas permiten eliminar progresivamente los coeficientes ubicados debajo de la diagonal principal.
- La sustitución hacia atrás permite obtener las incógnitas una vez finalizada la etapa de eliminación.
- Un pivote igual a cero requiere realizar un intercambio de filas antes de continuar.
- La implementación con intercambio de la fila siguiente funciona para el ejemplo estudiado, aunque no representa un algoritmo general de pivoteo parcial.
- Este método complementa los métodos iterativos de Jacobi y Gauss-Seidel estudiados en otros notebooks del repositorio.